In [4]:
from collections import defaultdict
import re

In [13]:
def parse_logs(file_path: str):
    # 按requestId返祖存储日志行
    log_blocks = defaultdict(list)

    # 正则表达式，格式为[reqId:xxx]
    request_id_pattern = re.compile(r'\[reqId:(\S+)\]')
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # 搜寻search该模式。如果是match则为匹配
            req_id_match = request_id_pattern.search(line)
            if req_id_match:
                req_id = req_id_match.group(1)
                log_blocks[req_id].append(line)
            else:
                # 没有reqId的日志，单独放入一个分类
                log_blocks["NO_REQ_ID"].append(line)

    return log_blocks

In [25]:
parse_logs('D://internship//sitech//开发者社区文档//test_info.txt')

defaultdict(list,
            {'req-123': ['[2023-10-25 14:30:12,456] [INFO] [reqId:req-123] UserService: Request received from user-456',
              '[2023-10-25 14:30:12,567] [DEBUG] [reqId:req-123] DB: Querying user profile...',
              '[2023-10-25 14:30:12,789] [ERROR] [reqId:req-123] UserService: DB timeout! Failed to get user.'],
             'req-124': ['[2023-10-25 14:30:13,001] [INFO] [reqId:req-124] PaymentService: Starting payment for order-789']})

In [14]:
def clean_and_extract_metadata(log_line: str):
    pattern = re.compile(
        r'\[(?P<timestamp>.+?)\] '
        r'\[(?P<level>\w+)\] '
        r'(?:\[reqId:\S+\] )?'
        r'(?P<service>\w+): '
        r'(?P<message>.*)'
    )

    match = pattern.match(log_line)
    if not match:
        return None

    log_data = match.groupdict()

    return {
        "raw_log": log_line,
        "message": log_data["message"],
        "timestamp": log_data["timestamp"],
        "level": log_data["level"],
        "service": log_data["service"]
    }

In [15]:
def build_log_block(request_id: str, log_lines: list):
    # List<Map<>>
    cleaned_logs = []
    for line in log_lines:
        cleaned = clean_and_extract_metadata(line)
        if cleaned:
            cleaned_logs.append(cleaned)

    if not cleaned_logs:
        return None

    cleaned_logs.sort(key=lambda log: log["timestamp"])
    block_text = " | ".join([log["message"] for log in cleaned_logs])

    first_log = cleaned_logs[0]
    last_log = cleaned_logs[-1]

    metadata = {
        "request_id": request_id,
        "start_time": first_log["timestamp"],
        "end_time": last_log["timestamp"],
        "services": list(set(log["service"] for log in cleaned_logs)),
        "log_level": list(set(log["level"] for log in cleaned_logs)),
    }

    return {
        "metadata": metadata,
        "text_for_embedding": block_text,
        "raw_logs": [log["raw_log"] for log in cleaned_logs]
    }

In [16]:
def process_log_file(file_path: str):
    grouped_logs = parse_logs(file_path)

    blocks = []
    for req_id, log_lines in grouped_logs.items():
        block = build_log_block(req_id, log_lines)
        if block:
            blocks.append(block)

    return blocks

In [1]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

C:\Users\24978\.conda\envs\llm\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [2]:
""" 初始化向量数据库 """
client = chromadb.PersistentClient(path="./chroma_data")
collection = client.get_or_create_collection(
    name="server_logs",
    metadata={"hnsw:space": "cosine"}
)

In [3]:

# 加载embedding模型。使用已有大embedding模型
model_path = 'D:/cs/projects/RAG/mooc/RAG_full_stack_course_notebooks/llm_app/gte-large-zh/'
# embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
embed_model = SentenceTransformer(model_path)

In [18]:
""" 将预处理后的日志块存入向量数据库 """
def store_log_blocks(log_blocks: list):
    # 分别存储log_id, 文档, 元数据
    ids, documents, metadatas = [], [], []

    # 对于单个日志块
    for block in log_blocks:
        # 设置日志id，起始时间+调用者的hash
        log_id = f"{block['metadata']['start_time']}-{hash(block['metadata']['services'][0])}"
        ids.append(log_id)
        documents.append(block['text_for_embedding'])
        metadatas.append({
            "request_id": block['metadata']['request_id'],
            "start_time": block['metadata']['start_time'],
            "service": block['metadata']['services'][0]
        })

    # 批量生成向量(CPU高效)
    embeddings = embed_model.encode(documents).tolist()

    # 存入向量数据库
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )

In [4]:
"""
混合检索日志
:param query: 自然语言查询
:param filters: 元数据过滤
:param top_k: 返回结果数量
"""
def search_logs(query: str, filters: dict = None, top_k:int = 5):
    # 1. 生成查询向量, 用于模糊查询
    query_embedding = embed_model.encode([query]).tolist()[0]

    # 2. 构建混合查询条件
    where_conditions = {}
    if filters:
        if "start_time" in filters:
            where_conditions["start_time"] = {"$gte": filters["start_time"]}
        if "service" in filters:
            where_conditions["service"] = {"$eq": filters["service"]}

    # 3. 执行检索: 元数据过滤 + 语义相似度
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where=where_conditions, # 元数据过滤
        include=["documents", "metadatas", "distances"] # 返回原始文本, 元数据, 相似度距离
    )

    # 4. 整理结果
    output = []
    for i in range(len(results["ids"][0])):
        output.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "score": results["distances"][0][i],
            "metadata": results["metadatas"][0][i]
        })

    return output



In [19]:
# 存储数据
file_path = 'D://internship//sitech//开发者社区文档//test_info.txt'
log_blocks = process_log_file(file_path)
store_log_blocks(log_blocks)


In [9]:
# 搜索（score越大越远）
search_logs(query="DB timeout", top_k=5)

[{'id': '2023-10-25 14:30:12,456--1395151406513750894',
  'text': 'Request received from user-456 | Querying user profile... | DB timeout! Failed to get user.',
  'score': 0.5327990557768352,
  'metadata': {'request_id': 'req-123',
   'service': 'DB',
   'start_time': '2023-10-25 14:30:12,456'}},
 {'id': '2023-10-25 14:30:12,456--7668613606286957039',
  'text': 'Request received from user-456 | Querying user profile... | DB timeout! Failed to get user.',
  'score': 0.5327990557768352,
  'metadata': {'request_id': 'req-123',
   'service': 'UserService',
   'start_time': '2023-10-25 14:30:12,456'}},
 {'id': '2023-10-25 14:30:12,456--7747406125837326656',
  'text': 'Request received from user-456 | Querying user profile... | DB timeout! Failed to get user.',
  'score': 0.5327990557768352,
  'metadata': {'request_id': 'req-123',
   'service': 'UserService',
   'start_time': '2023-10-25 14:30:12,456'}},
 {'id': '2023-10-25 14:30:13,001-4493339174568899481',
  'text': 'Starting payment for o

In [1]:
import modules.VectorDB as VectorDB
vector_db = VectorDB.VectorDB('D:/cs/projects/RAG/mooc/RAG_full_stack_course_notebooks/llm_app/gte-large-zh/')


C:\Users\24978\.conda\envs\llm\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [2]:
results = vector_db.search_logs(
    query="DB timeout",
    top_k=5
)
results

Add of existing embedding ID: 2023-10-25 14:30:12,456--7668613606286957039
Add of existing embedding ID: 2023-10-25 14:30:13,001--8932832465059432921
Add of existing embedding ID: 2023-10-25 14:30:12,456--7668613606286957039
Add of existing embedding ID: 2023-10-25 14:30:13,001--8932832465059432921
Add of existing embedding ID: 2023-10-25 14:30:12,456--7668613606286957039
Add of existing embedding ID: 2023-10-25 14:30:13,001--8932832465059432921


[{'id': '2023-10-25 14:30:12,456--1395151406513750894',
  'text': 'Request received from user-456 | Querying user profile... | DB timeout! Failed to get user.',
  'score': 0.5327990557768352,
  'metadata': {'request_id': 'req-123',
   'service': 'DB',
   'start_time': '2023-10-25 14:30:12,456'}},
 {'id': '2023-10-25 14:30:12,456--7668613606286957039',
  'text': 'Request received from user-456 | Querying user profile... | DB timeout! Failed to get user.',
  'score': 0.5327990557768352,
  'metadata': {'request_id': 'req-123',
   'service': 'UserService',
   'start_time': '2023-10-25 14:30:12,456'}},
 {'id': '2023-10-25 14:30:12,456--7747406125837326656',
  'text': 'Request received from user-456 | Querying user profile... | DB timeout! Failed to get user.',
  'score': 0.5327990557768352,
  'metadata': {'request_id': 'req-123',
   'service': 'UserService',
   'start_time': '2023-10-25 14:30:12,456'}},
 {'id': '2023-10-25 14:30:13,001-4493339174568899481',
  'text': 'Starting payment for o

In [ ]:
def debug():
    from sentence_transformers import SentenceTransformer
    import time

    print("开始加载模型")
    model = SentenceTransformer('D:/cs/projects/RAG/mooc/RAG_full_stack_course_notebooks/llm_app/gte-large-zh/')
    print("模型属性检查：")
    print(f"- 是否有encode方法：{hasattr(model, 'encode')}")
    print(f"- 模型设备: {getattr(model, 'device', '未设置')}")

    try:
        emb = model.encode("debug")
        print(f"嵌入向量形状:{emb.shape}")
    except Exception as e:
        print(f"推理失败，错误：{str(e)}")

